In [ ]:
from db import query
import pandas as pd

bonf = query("SELECT * FROM bonferroni_fdr_results WHERE feature_group = 'text'")
boot = query("SELECT * FROM bootstrap_results WHERE feature_group = 'text'")
wf   = query("SELECT * FROM walkforward_results WHERE feature_group = 'text'")

def subgroup(feat):
    if feat == 'sentiment_score': return 'sentiment'
    if feat.startswith('lda_topic'): return 'lda'
    return 'embeddings'

for df, method, col in [
    (bonf, 'Bonferroni', 'bonferroni_rejected'),
    (boot, 'Bootstrap',  'bootstrap_rejected'),
    (wf,   'Walk-Forward','wf_rejected'),
]:
    df['subgroup'] = df['feature'].apply(subgroup)
    print(f"\n{method}")
    print(df.groupby('subgroup').agg(
        total=('feature','count'),
        rejected=(col, 'sum'),
        rejection_rate=(col, 'mean')
    ).round(3))

In [ ]:
import matplotlib.pyplot as plt

bonf['subgroup'] = bonf['feature'].apply(subgroup)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, sg in zip(axes, ['sentiment', 'lda', 'embeddings']):
    subset = bonf[bonf['subgroup'] == sg]['correlation']
    ax.hist(subset, bins=30, edgecolor='black')
    ax.set_title(f'{sg} (n={len(subset)})')
    ax.set_xlabel('Pearson r with next_day_return')
plt.tight_layout()
plt.savefig('plots/text_correlation_distributions.png', dpi=150)
plt.show()

In [ ]:
embed = bonf[bonf['subgroup'] == 'embeddings'].copy()
embed['dim'] = embed['feature'].str.extract(r'embed_(\d+)').astype(int)
embed_sorted = embed.sort_values('p_value').head(20)
display(embed_sorted[['dim','correlation','p_value','bonferroni_rejected']])

In [ ]:
lda = bonf[bonf['subgroup'] == 'lda'][['feature','correlation','p_value','bonferroni_rejected']]
display(lda.sort_values('p_value'))

sentiment = bonf[bonf['subgroup'] == 'sentiment']
display(sentiment)